In [13]:
import os
import requests

DEEPSEEK_API_KEY = os.environ.get("DEEPSEEK_API_KEY", "")
BASE_URL = "https://api.deepseek.com/v1/chat/completions"  # deepseek的url

if not DEEPSEEK_API_KEY:
    raise ValueError("请在环境变量中设置 DEEPSEEK_API_KEY")

def get_completion(prompt, model="deepseek-chat", temperature=0):
    headers = {
        "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,
    }
    resp = requests.post(BASE_URL, headers=headers, json=payload)
    resp.raise_for_status()
    data = resp.json()
    # 具体取值方式看 DeepSeek 返回的 JSON 结构，一般类似：
    return data["choices"][0]["message"]["content"]

In [14]:
# 使用分隔符清晰地表示输入的不同部分

enterprise_text = """
某城投公司，主要从事市政基础设施建设和运营，所在城市为三线城市。
公司总资产 300 亿元，资产负债率约 72%，近三年营业收入在 40–45 亿元之间，
但经营性现金流波动较大，部分项目回款依赖财政补贴。
公司对外担保余额较高，对地方平台公司有连带责任，存在一定或有负债风险。
"""

prompt = f"""
你是一名商业银行公司金融条线的客户经理。

请严格根据 ``` 内的材料，完成以下任务：
1. 用 3 句话总结该公司的基本经营与财务状况；
2. 分别用 1 句话概括：主营业务风险、现金流风险、或有负债风险；
3. 禁止编造材料中没有的信息。

企业材料如下：
```{enterprise_text}```
"""

print(get_completion(prompt))

1.  该公司是一家位于三线城市的城投公司，核心业务为市政基础设施的建设和运营。公司资产规模达300亿元，但财务杠杆较高，资产负债率约为72%。近三年营业收入稳定在40至45亿元区间，但经营性现金流波动显著，部分项目回款需依赖财政资金。
2.  主营业务风险：部分项目回款依赖财政补贴，收入来源的确定性和及时性存在不确定性。
    现金流风险：公司经营性现金流波动较大，影响其自身经营的资金平衡能力。
    或有负债风险：公司对外担保余额较高，且主要为对地方平台公司的连带责任担保，存在代偿风险。


In [8]:
# 结构化输出：将企业信息提炼为 JSON

enterprise_text = """
某制造业企业，主营消费电子代工，客户集中度较高，
前五大客户收入占比约 70%。近三年营收年均增速 15%，
但应收账款占总资产比例持续上升，存货周转放缓。
企业有银行借款 5 亿元，资产负债率约 65%。
"""

prompt = f"""
请根据 ``` 内的企业材料，输出 JSON 格式的结构化摘要。

必须包含以下字段（字段名用英文）：
- company_name：企业名称（如未知可写 "未知"）
- industry：行业
- revenue_trend：营收趋势（1 句话）
- cashflow_status：现金流与营运资本状况（1–2 句话）
- main_risks：主要风险点（字符串列表）
- credit_decision：授信建议，取值只能是 "approve" 或 "reject"

只输出 JSON，不要多余说明文字。

```{enterprise_text}```
"""

print(get_completion(prompt))

{
  "company_name": "未知",
  "industry": "消费电子代工",
  "revenue_trend": "近三年营收年均增速15%。",
  "cashflow_status": "应收账款占总资产比例持续上升，存货周转放缓，资产负债率约65%。",
  "main_risks": ["客户集中度高，前五大客户收入占比约70%", "应收账款占比上升", "存货周转放缓", "资产负债率较高（65%）"],
  "credit_decision": "reject"
}


In [15]:
# 条件检查：判断材料中是否出现“还款计划”相关内容

text_without_plan = """
本项目拟用于扩建产能，总投资 5 亿元，其中银行贷款 3 亿元。
预计投产后每年新增营业收入 4 亿元，净利润 3 千万元。
企业计划通过自有资金和外部融资共同解决资金缺口。
"""

prompt = f"""
你将获得一段项目融资材料（用 ``` 括起来）。

请按以下要求处理：
1. 先判断文本中是否出现了明确的“还款安排/还款计划”，例如：
   - 还款期限（多少年）
   - 本金/利息的偿还方式（等额本息、按季付息等）
2. 如果存在，请用 2–3 句话摘录并概括还款计划；
3. 如果不存在，请只输出：未提供还款计划。

```{text_without_plan}```
"""

print(get_completion(prompt))


未提供还款计划。


In [16]:
# Few-shot：模仿信贷审批意见风格

prompt = """
你的任务是模仿示例中的“信贷审批意见”写作风格。

【示例 1】
<企业情况>：
区域连锁超市，门店分布较为分散，现金流稳定，
有自有物业用于抵押，历史还款记录良好。

<审批意见示例>：
综合考虑企业持续经营能力、抵押物价值及历史履约情况，
建议给予适度流动资金贷款授信，期限不超过 1 年，
并通过定期报送经营数据和库存情况实现贷后跟踪。

【示例 2】
<企业情况>：
某房地产开发企业，项目去化进度不及预期，库存较多，
有较大到期债务压力，短期流动性紧张。

<审批意见示例>：
鉴于公司负债率较高、现金流紧张且项目销售存在不确定性，
建议暂不新增授信，并加强对存量风险敞口和项目现金回笼情况的监测。

【待处理企业】
<企业情况>：
某光伏组件制造企业，近三年收入保持 20% 左右增速，
但应收账款占比偏高，部分客户集中度较高，
主要抵押物为厂房及生产线设备。

请参照上述“审批意见示例”的行文结构和语气，
为待处理企业撰写一段信贷审批意见（不超过 200 字）。
"""

print(get_completion(prompt))


综合考虑企业成长性、应收账款质量及抵押物变现能力，建议给予适度授信支持，但需关注客户集中风险。可提供流动资金贷款，期限不超过1年，并追加主要设备抵押。授信后应要求企业定期提供应收账款账龄明细及主要客户回款情况，加强现金流监测。


In [17]:
# 指定完成任务的步骤：上市公司财报分析

financial_text = """
某上市公司，主营家用电器销售，近三年营收分别为 100/120/132 亿元，
归母净利润分别为 5/6/6.2 亿元，净利率保持在约 4.5%–4.7%。
经营活动现金流量净额整体为正，但波动较大。
公司有部分短期借款和一年内到期的非流动负债，资产负债率约 60%。
"""

prompt = f"""
请严格按照以下步骤，对 ``` 内的财报摘要进行分析：

步骤 1：用 3–4 句话总结公司在营收、利润和现金流方面的整体变化趋势；
步骤 2：列出 3 个你认为最关键的财务指标（可以从增长率、净利率、资产负债率、经营现金流等中选），
        并说明每个指标对该企业意味着什么；
步骤 3：从“盈利能力 / 偿债能力 / 成长性”三个角度分别写一段分析（每段 3–4 句话）；
步骤 4：最后用一句话给出“是否建议进一步尽调该公司”的结论（只能选 建议/暂缓/不建议），并简要说明理由。

请按“步骤 1 / 步骤 2 / 步骤 3 / 步骤 4”的标题格式输出结果。

```{financial_text}```
"""

print(get_completion(prompt))

**步骤 1：整体变化趋势总结**
公司在过去三年营收保持增长，但增速有所放缓（从20%降至10%）。归母净利润同步增长，但增速同样放缓，且净利率水平稳定在较低区间。经营活动现金流虽整体为正，但波动较大，表明公司主营业务的现金回收稳定性有待加强。

**步骤 2：关键财务指标及意义**
1.  **营收增长率（从20%降至10%）**：该指标显示公司收入增长动能正在减弱，可能面临市场饱和或竞争加剧的压力，未来成长性需要关注。
2.  **净利率（约4.5%-4.7%）**：该指标稳定但处于较低水平，说明公司的盈利能力不强，成本费用控制或产品附加值提升空间有限，利润对收入增长的依赖度高。
3.  **资产负债率（约60%）**：该指标处于相对偏高水平，结合公司存在短期借款和一年内到期负债，表明公司财务杠杆运用较多，偿债压力与财务风险需密切关注。

**步骤 3：分角度分析**
*   **盈利能力**：公司盈利能力稳定但偏弱。净利率常年维持在4.5%-4.7%的较低水平，表明其产品或服务的附加值不高，成本控制或定价能力有限。利润增长主要依赖营收规模的扩大，而非盈利质量的提升。在营收增速放缓的背景下，未来利润增长可能面临挑战。
*   **偿债能力**：公司偿债能力存在一定压力。约60%的资产负债率表明财务结构偏激进，依赖负债经营。同时，存在短期借款和一年内到期的非流动负债，意味着近期有明确的偿债资金需求。经营现金流波动较大，可能影响其按期偿债的稳定性，需警惕流动性风险。
*   **成长性**：公司成长性呈现放缓趋势。营收增速从20%显著下降至10%，显示市场扩张或收入增长的动力正在减弱。在家电行业可能进入存量竞争的背景下，公司尚未展现出通过提升盈利效率（净利率未改善）来驱动高质量增长的能力，未来持续成长面临不确定性。

**步骤 4：尽调建议**
**建议进一步尽调该公司。** 理由：公司基本面（持续盈利、正现金流）尚可，但关键风险点（增长放缓、盈利偏弱、偿债压力）明确，需要通过深入尽调来评估其风险的具体成因、可控性以及未来的改善潜力。


In [18]:
# 改进版：指定输出格式的财务分析模板

financial_text = """
某民营医药零售企业，近三年收入从 30 亿元增长至 45 亿元，
同店销售稳步增长，新开门店带来增量，但期间费用率有所上升。
经营性现金流整体为正，但库存金额有上升趋势。
企业有部分短期银行借款和租赁负债，整体资产负债率约 55%。
"""

prompt = f"""
请根据 ``` 内的材料，按以下固定格式输出分析结果（格式不要改动标题）：

【一、公司概况】
<用 2–3 句话介绍公司业务与最近三年变化>

【二、关键财务指标】
- 指标 1：
- 指标 2：
- 指标 3：

【三、能力分析】
1. 盈利能力：
2. 偿债能力：
3. 成长性：

【四、综合判断】
<用 1–2 句话给出是否建议进一步尽调，并说明理由>

以下为公司财务摘要：
```{financial_text}```
"""

print(get_completion(prompt))

【一、公司概况】
该公司是一家民营医药零售企业，主营业务为药品及健康产品的零售。近三年，公司收入从30亿元增长至45亿元，主要通过同店销售稳步增长和积极开设新门店实现规模扩张。

【二、关键财务指标】
- 指标 1：近三年收入从30亿元增长至45亿元，年复合增长率显著。
- 指标 2：整体资产负债率约55%，处于行业中等水平。
- 指标 3：经营性现金流整体为正，但库存金额有上升趋势。

【三、能力分析】
1. 盈利能力：收入增长强劲，显示市场拓展和运营有效，但期间费用率上升可能对净利润率造成压力。
2. 偿债能力：资产负债率约55%，存在部分短期银行借款和租赁负债，偿债压力尚可但需关注流动性管理。
3. 成长性：收入持续增长，同店销售稳步提升且新店贡献增量，显示良好的市场成长性和扩张能力。

【四、综合判断】
建议进一步尽调。理由是公司展现出良好的收入成长性和市场扩张能力，但需重点关注期间费用率上升的原因、库存增长对现金流的具体影响以及债务结构的详细情况，以全面评估其盈利质量和财务风险。


In [19]:
# 让模型先解题再判学生对错：理财产品综合收益率

prompt = """
请按照以下步骤判断学生的解答是否正确，并使用 LaTeX 数学公式渲染所有推导过程。

【步骤要求】
1. 你先独立完成这道题的计算和推导；
2. 写出你的推导过程和正确的计算公式；
3. 将你的解答与学生的解答进行对比；
4. 判断学生解答是否正确，并说明原因。

【公式渲染要求】
- 行内公式用 $...$
- 独立成行的公式用 $$...$$
- 多行推导用：
  $$\\begin{aligned}
  ...
  \\end{aligned}$$
- 请确保输出可以在 Markdown 中正确渲染。

【题目】
某一年期理财产品，名义预期年化收益率为 6%。银行按认购金额收取 1% 的管理费，销售渠道额外收取 0.5% 的销售服务费，均在产品购买时一次性扣除。请写出投资者实际到手收益率 r 的计算公式。

【学生解答】
r = 6% - 1% = 5%。

【输出格式】
请严格按照下列格式输出：

问题：
学生的解答：
你的解答和计算步骤：
最终公式（单独展示）：
学生的解答是否正确（正确/错误）：
原因说明（严格基于数学逻辑）：
"""

print(get_completion(prompt))

问题：  
某一年期理财产品，名义预期年化收益率为 6%。银行按认购金额收取 1% 的管理费，销售渠道额外收取 0.5% 的销售服务费，均在产品购买时一次性扣除。请写出投资者实际到手收益率 r 的计算公式。

学生的解答：  
r = 6% - 1% = 5%。

你的解答和计算步骤：  
设认购金额为 $P$。  
购买时一次性扣除的管理费和销售服务费合计为 $1\% + 0.5\% = 1.5\%$，因此实际投入本金为  
$$P_{\text{实}} = P \times (1 - 1.5\%) = P \times 0.985.$$  
一年后，按名义年化收益率 6% 计算，产品到期价值为  
$$P \times (1 + 6\%) = P \times 1.06.$$  
投资者实际到手金额为 $P \times 1.06$（因为费用已在购买时扣除，到期不再扣）。  
实际到手收益率 $r$ 定义为  
$$r = \frac{\text{到手金额} - \text{实际投入本金}}{\text{实际投入本金}}.$$  
代入得  
$$\begin{aligned}  
r &= \frac{P \times 1.06 - P \times 0.985}{P \times 0.985} \\  
  &= \frac{1.06 - 0.985}{0.985} \\  
  &= \frac{0.075}{0.985} \\  
  &\approx 0.076142.  
\end{aligned}$$  
即 $r \approx 7.6142\%$。

最终公式（单独展示）：  
$$r = \frac{1 + 6\%}{1 - (1\% + 0.5\%)} - 1 = \frac{1.06}{0.985} - 1.$$

学生的解答是否正确（正确/错误）：  
错误。

原因说明（严格基于数学逻辑）：  
学生的解答错误地将费用直接从名义收益率中减去（$6\% - 1\% = 5\%$），这忽略了两个关键点：  
1. 费用是在购买时一次性扣除，影响的是实际投入本金，而不是直接减少名义收益；  
2. 销售服务费 $0.5\%$ 也被忽略。  
正确的计算方法应基于实际投入本金和到期到手金额，按收益率定义 $r = \frac{\text{到手

In [20]:
# 如果以上公式形式不够直观，那就把公式用文本形式输出

prompt = """
请按照以下步骤解题并判断学生答案是否正确，并用“纯文本公式”表达所有计算过程。

【步骤要求】
1. 你先独立完成整道题的数学推导与计算；
2. 写出清晰的计算步骤；
3. 将你的解答与学生的解答进行对比；
4. 判断学生解答是否正确；
5. 说明原因（严格基于数学逻辑）。

【公式书写要求】
- 禁止使用任何 LaTeX 语法（不要用 $、$$、\frac、^{} 等）；
- 所有分数写成 (分子)/(分母) 的形式；
- 乘法可以写成 "*" 或 "×"；
- 示例：r = (1 + 6%) / (1 - 1.5%) - 1

【题目】
某一年期理财产品，名义预期年化收益率为 6%。银行按认购金额收取 1% 的管理费，销售渠道额外收取 0.5% 的销售服务费，均在产品购买时一次性扣除。请写出投资者实际到手收益率 r 的计算公式。

【学生解答】
r = 6% - 1% = 5%。

【输出格式】
请严格按照以下格式输出：

问题：
学生的解答：
你的解答和计算步骤：
最终公式：
学生的解答是否正确（正确/错误）：
原因说明（严格基于数学逻辑）：
"""

print(get_completion(prompt))

问题：  
某一年期理财产品，名义预期年化收益率为 6%。银行按认购金额收取 1% 的管理费，销售渠道额外收取 0.5% 的销售服务费，均在产品购买时一次性扣除。请写出投资者实际到手收益率 r 的计算公式。  

学生的解答：  
r = 6% - 1% = 5%。  

你的解答和计算步骤：  
1. 设认购金额为 P。  
2. 购买时扣除管理费 1% 和销售服务费 0.5%，总扣除比例为 1% + 0.5% = 1.5%。  
3. 实际投入本金为 P * (1 - 1.5%) = P * (1 - 0.015) = P * 0.985。  
4. 名义年化收益率为 6%，一年后名义收益为 P * 6% = P * 0.06。  
5. 一年后总金额为 P + P * 6% = P * (1 + 0.06) = P * 1.06。  
6. 实际到手收益率 r 定义为 (一年后总金额 - 实际投入本金) / 实际投入本金。  
7. 代入：r = (P * 1.06 - P * 0.985) / (P * 0.985)。  
8. 化简：r = (1.06 - 0.985) / 0.985 = 0.075 / 0.985。  
9. 用百分数表示：r = (0.075 / 0.985) * 100% ≈ 7.614%。  
10. 写成公式形式：r = (1 + 6%) / (1 - 1.5%) - 1。  

最终公式：  
r = (1 + 6%) / (1 - 1.5%) - 1  

学生的解答是否正确（正确/错误）：  
错误  

原因说明（严格基于数学逻辑）：  
学生的解答只从名义收益率中减去管理费 1%，忽略了销售服务费 0.5%，并且错误地将费用直接从收益率中扣除。实际收益率计算应考虑费用在购买时从本金中扣除，影响实际投入本金，再基于实际投入本金计算收益。正确的公式应为 r = (1 + 名义收益率) / (1 - 总费用率) - 1，其中总费用率 = 管理费率 + 销售服务费率。


In [28]:
# 幻觉示例 
prompt = """ 
 请说明中国银保监会《关于商业银行绿色信贷风险权重优惠指引》第 23 条第 4 款 
 对“绿色项目风险权重 25%”的具体规定和适用条件。
 
 """ 
print(get_completion(prompt))

好的，我们来详细解读中国银保监会（现为国家金融监督管理总局）《关于商业银行绿色信贷风险权重优惠指引》中关于“绿色项目风险权重25%”的核心规定。

首先需要明确，您提到的 **“第23条第4款”** 是一个假设性的条款编号。在实际的《商业银行资本管理办法（试行）》及其后续修订和配套政策中，**并没有一个独立的、名为《关于商业银行绿色信贷风险权重优惠指引》的文件直接规定“绿色项目风险权重25%”**。

然而，您所询问的 **“绿色项目风险权重25%”** 这一核心政策内容是真实存在的，它体现在 **《商业银行资本管理办法（试行）》** 的附件《权重法表内资产风险权重、表外项目信用转换系数及合格信用风险缓释工具》中，具体是针对 **“对公共部门实体（PSEs）的债权”** 这一项下的特殊规定。

下面，我将根据现行有效的监管规定，为您详细说明“绿色项目风险权重25%”的具体规定和适用条件。

### 核心规定

在《商业银行资本管理办法（试行）》中，对一般企业的债权风险权重为**100%**。但对于符合条件的绿色信贷，可以享受**75%** 的风险权重优惠。您提到的 **“25%”** 风险权重，实际上是一个更优惠的级别，其适用对象有严格的限定。

**1. 25%风险权重的具体规定：**

*   **适用对象**：商业银行对 **“公共部门实体”** 的债权。
*   **具体条款**：在《权重法》下，对“公共部门实体（PSEs）”的债权，一般适用20%或50%的风险权重。但其中有一项特殊规定：
    > **“对已抵押给商业银行的、已出租的公共租赁住房（包括公租房、廉租房、经济适用房等）的债权”，其风险权重为25%。**

**2. 75%风险权重的具体规定（更普遍的绿色信贷优惠）：**

*   **适用对象**：商业银行对 **“一般公司”** 的债权。
*   **具体条款**：对于符合标准的 **“绿色信贷”** ，其风险权重可以从标准的100%降至**75%**。
    > 这是目前商业银行在绿色信贷领域应用最广泛的风险权重优惠。

### 适用条件

无论是25%还是75%的优惠风险权重，都需要满足严格的认定条件。核心在于 **“项目”** 和 **“资金用途”** 必须符合官方标准。

#### 对于“25%风险权重”（公共租赁住房）：

1.

In [16]:
# 幻觉示例

prompt = """
请说明中国银保监会《关于商业银行绿色信贷风险权重优惠指引》第 23 条第 4 款
对“绿色项目风险权重 25%”的具体规定和适用条件。

（注意：这个文件和条款是虚构的，用于测试模型是否会编造内容。）
"""

print(get_completion(prompt))

您提出的问题非常敏锐，并且您指出的虚构性是正确的。这是一个非常好的测试案例。

首先，我需要明确指出：**中国银保监会（现已成为国家金融监督管理总局）并未发布过名为《关于商业银行绿色信贷风险权重优惠指引》的文件，其中也自然不存在所谓的“第23条第4款”关于“绿色项目风险权重25%”的规定。**

这是一个虚构的条款。因此，我将无法提供该“具体规定”的真实内容。

然而，为了回应您的问题，并展示一个负责任的AI应如何应对此类情况，我将从以下两个方面进行说明：

1.  **明确指出信息的虚构性并解释原因。**
2.  **基于中国现行的真实金融监管框架，解释“风险权重”和“绿色信贷”的相关概念，以说明为何这样的条款在逻辑上是可能被讨论的，但现实中并不存在。**

---

### 1. 对虚构条款的回应与分析

**核心结论：该条款不存在。**

*   **监管机构变更：** 原中国银行保险监督管理委员会（银保监会）已于2023年与中国人民银行对金融控股公司等机构的监管职责进行整合，组建为**国家金融监督管理总局（NFRA）**。因此，引用“银保监会”发布的新规需要格外注意时效性。
*   **文件不存在：** 在官方渠道（如NFRA官网）检索，不存在上述文件名称。
*   **具体条款不存在：** 最关键的是，中国目前的《商业银行资本管理办法（试行）》及其后续修订版本中，**并未对绿色信贷项目普遍给予“风险权重25%”的优惠**。

**为什么这个虚构条款听起来“合理”？**
因为它触及了中国绿色金融发展中的一个核心讨论议题：**是否应该通过降低资本要求来激励银行投放绿色信贷。**

### 2. 基于真实监管框架的解释

尽管该条款是虚构的，但我们可以通过了解真实规则来理解其背景。

#### （1）什么是“风险权重”？

*   根据《商业银行资本管理办法》，银行需要为其持有的资产（如发放的贷款）预留一定比例的资本，以覆盖潜在的损失风险。这个比例就是**资本充足率**计算中的关键因子。
*   “风险权重”就是衡量不同资产风险水平的系数。权重越低，意味着银行需要为此预留的资本就越少，从而激励银行更多地投放此类贷款。
*   例如，对中央政府的债权风险权重为0%，对一般企业的债权风险权重为100%。如果一笔贷款的风险权重是75%，就意味着银行需要预留的资本是10

In [18]:
# 减少幻觉：先抽取原文，再基于原文概括（绿色金融政策示例）

policy_text = """
《某省关于金融支持绿色低碳高质量发展的实施意见》节选：
金融机构要加大对绿色信贷、绿色债券等绿色金融产品的支持力度，
合理控制对高碳排放行业的授信投向和余额增速，
稳妥有序推进存量高碳资产的绿色低碳转型。
鼓励上市公司和债券发行人披露环境信息和碳排放信息，
引导更多社会资金参与绿色投资。
"""

prompt = f"""
你是一名研究绿色金融的分析师，请严格按照两步作答：

第 1 步：从 ``` 内的原文中，逐条摘录与“绿色金融支持方向”直接相关的句子，
          每条前面加上“原文要点 - ”，不要改写内容；
第 2 步：仅基于你刚才摘录的原文要点，用 3–4 句话总结本文件对绿色金融支持方向的主要要求，
          禁止添加原文中没有出现的新内容。

原文如下：
```{policy_text}```
"""
print(get_completion(prompt))

第 1 步：
原文要点 - 金融机构要加大对绿色信贷、绿色债券等绿色金融产品的支持力度，
原文要点 - 合理控制对高碳排放行业的授信投向和余额增速，
原文要点 - 稳妥有序推进存量高碳资产的绿色低碳转型。
原文要点 - 鼓励上市公司和债券发行人披露环境信息和碳排放信息，
原文要点 - 引导更多社会资金参与绿色投资。

第 2 步：
本文件要求绿色金融应重点支持绿色信贷、绿色债券等产品的发展。同时，需要限制对高碳排放行业的信贷投入，并推动现有高碳资产向绿色低碳转型。此外，还要求通过鼓励环境信息披露来引导社会资金更多地投向绿色领域。


In [19]:
# 关于反斜杠和换行的简单演示

prompt = (
    "请用两句话解释：在 Python 字符串里使用反斜杠 \\"
    " 断行与在 ChatGPT 直接输入长文本时，有什么区别？"
)

print(get_completion(prompt))

在Python字符串中，反斜杠`\`作为续行符会将物理行连接为逻辑行，实际存储时不会包含换行符。而在ChatGPT输入长文本时，直接换行会保留段落结构并作为实际内容处理，系统会自动适应文本长度无需手动断行。


In [29]:
# 情感分析
# ============ 模块 1：最简单的情感分析 ============

def sentiment_simple(text: str):
    """
    对单条中文文本做情感分类：正向 / 中性 / 负向
    返回模型的原始回答（字符串）。
    """
    prompt = f"""
你是一个中文情感分析助手，请判断下面这句话的情感倾向。

要求：
1. 只输出三个词之一：正向 / 中性 / 负向
2. 不要输出解释。

文本：
{text}
"""
    return get_completion(prompt)

# 示例
print(sentiment_simple("这只股票今天涨得我很满意。"))
print(sentiment_simple("这家公司业绩一般，没有什么亮点。"))
print(sentiment_simple("这个项目完全超出预算，简直一塌糊涂。"))

正向
负向
负向


In [30]:
# ============ 模块 2：情感 + 解释 ============

def sentiment_with_reason(text: str):
    """
    返回情感标签 + 简短原因说明
    """
    prompt = f"""
你是一个中文情感分析助手，请分析下面这段话的情感倾向，并给出简要原因。

输出格式固定为：
情感：正向 / 中性 / 负向
原因：一句话解释原因

文本：
{text}
"""
    return get_completion(prompt)

# 示例
print(sentiment_with_reason("公司盈利能力持续改善，未来发展值得期待。"))

情感：正向
原因：文本提到公司盈利能力持续改善和未来发展值得期待，表达了积极乐观的态度。


In [36]:
# ============ 模块 3：批量情感分析 ============

def batch_sentiment(text_list):
    """
    对一组文本做情感分析（简单版，使用模块 1）。
    返回一个列表，对应每条文本的情感标签。
    """
    results = []
    for t in text_list:
        label = sentiment_simple(t)
        results.append({"text": t, "sentiment": label})
    return results

# 示例
texts = [
    "这家银行的不良率在下降，资产质量稳中向好。",
    "政策环境存在不确定性，投资需要谨慎。",
    "公司连续几个季度亏损，现金流压力较大。"
]
for item in batch_sentiment(texts):
    print(item)

{'text': '这家银行的不良率在下降，资产质量稳中向好。', 'sentiment': '正向'}
{'text': '政策环境存在不确定性，投资需要谨慎。', 'sentiment': '负向'}
{'text': '公司连续几个季度亏损，现金流压力较大。', 'sentiment': '负向'}


In [40]:
# ============ 模块 4：金融文本情感分析（针对个股） ============

def sentiment_for_stock(text: str, stock_name: str):
    """
    针对某只股票（或公司）做情感分析：
    - 总体情感
    - 对股价影响
    - 关注点
    """
    prompt = f"""
你是一名证券分析师，请对下面这段与「{stock_name}」相关的新闻或研报内容进行情感分析。

任务：
1. 判断对 {stock_name} 的总体情感倾向：正向 / 中性 / 负向（三选一）；
2. 简要说明理由（2–3 句话）；
3. 说明你认为这段文本对股价短期影响的方向：利好 / 中性 / 利空（三选一）。

输出格式：
总体情感：正向 / 中性 / 负向
短期股价影响：利好 / 中性 / 利空
原因说明：若干句中文说明

文本：
{text}
"""
    return get_completion(prompt)

# 示例
news = "公司公布三季报，营收和净利润均超市场预期，并上调全年指引。"
print(sentiment_for_stock(news, "某某科技股份"))

总体情感：正向
短期股价影响：利好
原因说明：文本明确提到公司营收和净利润均超市场预期，这直接反映了公司良好的经营表现；同时公司上调全年指引，进一步增强了市场对其未来业绩的信心。这些积极因素通常会吸引投资者关注，对股价形成短期支撑。


In [34]:
# ============ 模块 5：课堂小练习 ============

def sentiment_homework(text: str):
    """
    学生练习用：让学生自己修改 prompt 中的“情感维度”，
    比如：从“正向/中性/负向”改为“乐观/谨慎/悲观”等。
    """
    prompt = f"""
你是一名宏观与金融市场分析师，请判断下面这段市场评论的情绪倾向。

请在以下三种中选择一种输出：
- 乐观
- 中性
- 悲观

只输出上述三个词之一，不要解释。

评论内容：
{text}
"""
    return get_completion(prompt)

print(sentiment_homework("美联储释放出更加鸽派的信号，新兴市场资产有望迎来资金流入。"))

乐观
